# Cuaderno E · Calor urbano

**LlajtaLab · NASA Space Apps 2026 Cochabamba · Enfoque 1**

Con MODIS Terra solo podemos medir el calor hasta 2019 (por el cambio de órbita). Este cuaderno lo resuelve y agrega tres medidas de calor a `tendencias_distritos.json`:

| Medida | Fuente | Periodo | Por qué importa |
|---|---|---|---|
| Calor del suelo de día, detallado | Landsat 8 y 9 (NASA/USGS), 100 m | 2013–2025, época seca | Landsat pasa siempre a la misma hora: no tiene el problema de Terra y llega hasta hoy |
| Calor del suelo de noche | MODIS Aqua (NASA), 1 km | 2003–2021 | Las islas de calor nocturnas afectan el descanso y la salud |
| Días de calor extremo por año | MERRA-2 (NASA), toda la ciudad | 2001–2025 | Cuenta los días más calurosos que lo normal |

**Sube a este Colab:**
1. `distritos_indicadores_ambientales_2024.geojson`
2. `tendencias_distritos.json` **el que generó el Cuaderno A** (el de `public/data/` ya actualizado)

Cada celda guarda un CSV de respaldo. Si Colab se corta, sube ese CSV y la celda no se repite.

## Celda E1 · Preparación

In [1]:
!pip install -q earthengine-api pymannkendall
import ee, json, time, itertools, os, pandas as pd, numpy as np, pymannkendall as mk
from scipy import stats
ee.Authenticate(); ee.Initialize(project='microclimas-cochabamba')

gj = json.load(open('/content/distritos_indicadores_ambientales_2024.geojson', encoding='utf-8'))
CAMPO_ID, CAMPO_NOMBRE = 'codigo_distrito', 'nombre'
distritos = ee.FeatureCollection(gj).select([CAMPO_ID, CAMPO_NOMBRE])
municipio = distritos.geometry().dissolve(100)
tj = json.load(open('/content/tendencias_distritos.json', encoding='utf-8'))
print('Variables que ya tiene el JSON:', list(tj['distritos'][0]['tendencias'].keys()))

def clave(v):
    return str(int(float(v)))

def reintentar(fun, etiqueta):
    for intento in range(6):
        try:
            return fun()
        except Exception as e:
            print(etiqueta, 'reintento:', str(e)[:80]); time.sleep(20 * (intento + 1))
    raise RuntimeError(f'No se pudo: {etiqueta}')

def analizar_mensual(serie):          # DataFrame con columnas fecha y valor
    s = serie.set_index('fecha').asfreq('MS'); v = s.iloc[:, 0].astype(float)
    r = mk.seasonal_test(v.values, period=12)
    return {'sen_pendiente_anual': round(float(r.slope), 4), 'mk_p': round(float(r.p), 4),
            'mk_tendencia': r.trend, 'significativa': bool(r.p < 0.05), 'meses_validos': int(v.notna().sum())}

def analizar_anual(anios, valores):   # una medición por año
    s = pd.Series(valores, index=anios, dtype=float).dropna()
    pares = [(s.iloc[j] - s.iloc[i]) / (s.index[j] - s.index[i]) for i, j in itertools.combinations(range(len(s)), 2)]
    r = mk.original_test(s.values)
    lr = stats.linregress(s.index.values.astype(float), s.values)
    tc = stats.t.ppf(0.975, len(s) - 2)
    return {'sen_pendiente_anual': round(float(np.median(pares)), 4), 'mk_p': round(float(r.p), 4),
            'mk_tendencia': r.trend, 'significativa': bool(r.p < 0.05),
            'lineal_pendiente_anual': round(float(lr.slope), 4),
            'ic95': [round(float(lr.slope - tc * lr.stderr), 4), round(float(lr.slope + tc * lr.stderr), 4)],
            'anios_validos': int(len(s))}

Variables que ya tiene el JSON: ['lst', 'ndvi', 'aod', 't2m']


In [9]:
import os
if os.path.exists('/content/calor_landsat.csv'):
    os.remove('/content/calor_landsat.csv')
    print('Respaldo vacío borrado')

Respaldo vacío borrado


## Celda E2 · Calor del suelo con Landsat, época seca 2013–2025

Para cada año: mediana de todas las escenas sin nubes de junio a octubre, así cada año se compara con la misma época.

**Revisa:** valores entre unos 25 y 45 °C (es el suelo cerca del mediodía en época seca). Si algún año tiene muy pocas escenas, avisa.

In [10]:
ARCH_L = '/content/calor_landsat.csv'
if os.path.exists(ARCH_L):
    L = pd.read_csv(ARCH_L, dtype={'id': str}); print('Usando', ARCH_L)
else:
    def st_landsat(img):
        sin_nubes = img.select('QA_PIXEL').bitwiseAnd(0b11111).eq(0)   # sin relleno, nubes, cirros ni sombras
        return img.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15).updateMask(sin_nubes)

    landsat = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
               .filterBounds(municipio))
    filas = []
    for y in range(2013, 2026):
        c = landsat.filterDate(f'{y}-06-01', f'{y}-11-01')
        img = c.map(st_landsat).median().rename('lst')
        res = reintentar(lambda: img.reduceRegions(collection=distritos, reducer=ee.Reducer.mean(), scale=100, tileScale=4)
                         .map(lambda f: f.setGeometry(None)).getInfo()['features'], y)
        n = c.size().getInfo()
        for f in res:
            filas.append({'id': clave(f['properties'][CAMPO_ID]), 'anio': y, 'lst': f['properties'].get('mean'), 'escenas': n})
        print(y, '·', n, 'escenas')
    L = pd.DataFrame(filas)
    L.to_csv(ARCH_L, index=False)

L['lst_rel'] = L['lst'] - L.groupby('anio')['lst'].transform('mean')   # comparado con el promedio del valle ese año
print(L.pivot(index='anio', columns='id', values='lst').round(1))

2013 · 17 escenas
2014 · 19 escenas
2015 · 19 escenas
2016 · 19 escenas
2017 · 19 escenas
2018 · 19 escenas
2019 · 19 escenas
2020 · 19 escenas
2021 · 18 escenas
2022 · 39 escenas
2023 · 38 escenas
2024 · 38 escenas
2025 · 39 escenas
id       1    10    11    12    13    14    15     2     3     4     5     6  \
anio                                                                           
2013  25.6  27.4  26.4  27.1  19.8  26.7  31.2  26.5  28.3  29.2  30.4  26.2   
2014  26.6  28.3  26.9  27.6  18.8  26.8  29.8  26.5  27.8  28.3  30.5  25.8   
2015  27.1  28.7  27.8  27.0  19.9  28.9  33.7  26.2  28.6  28.9  31.1  27.7   
2016  29.2  31.9  29.2  31.4  24.8  30.0  32.6  31.2  32.3  32.8  34.7  30.5   
2017  25.4  27.2  26.4  26.2  19.9  26.1  29.9  24.8  25.7  27.5  31.1  26.0   
2018  23.4  25.5  24.5  24.9  17.7  23.3  27.7  24.6  25.0  25.4  26.9  23.7   
2019  26.8  27.5  27.4  25.2  19.7  28.5  32.5  26.3  27.7  26.4  30.7  27.1   
2020  24.0  25.8  25.4  25.5  19.3  26.0  30.5

## Celda E3 · Calor del suelo de noche (MODIS Aqua, 2003–2021)

In [11]:
ARCH_N = '/content/calor_noche.csv'
if os.path.exists(ARCH_N):
    N = pd.read_csv(ARCH_N, dtype={'id': str}, parse_dates=['fecha']); print('Usando', ARCH_N)
else:
    noche_col = ee.ImageCollection('MODIS/061/MYD11A2').select('LST_Night_1km')
    vacia = ee.Image.constant(0).updateMask(ee.Image.constant(0))

    def img_anual_noche(y):
        b = []
        for m in range(1, 13):
            ini = ee.Date.fromYMD(y, m, 1); c = noche_col.filterDate(ini, ini.advance(1, 'month'))
            b.append(ee.Image(ee.Algorithms.If(c.size().gt(0), c.mean(), vacia)).multiply(0.02).subtract(273.15).rename(f'n_{m:02d}'))
        return ee.Image.cat(b)

    filas = []
    for y in range(2003, 2022):
        res = reintentar(lambda: img_anual_noche(y).reduceRegions(collection=distritos, reducer=ee.Reducer.mean(), scale=1000, tileScale=4)
                         .map(lambda f: f.setGeometry(None)).getInfo()['features'], y)
        for f in res:
            for m in range(1, 13):
                filas.append({'id': clave(f['properties'][CAMPO_ID]), 'fecha': f'{y}-{m:02d}', 'lst_noche': f['properties'].get(f'n_{m:02d}')})
        print('Listo', y)
    N = pd.DataFrame(filas); N['fecha'] = pd.to_datetime(N['fecha'])
    N.to_csv(ARCH_N, index=False)

print(N.groupby('id')['lst_noche'].mean().round(1))

Usando /content/calor_noche.csv
id
1     10.1
10    10.4
11    10.6
12    10.3
13     5.4
14    10.4
15     9.5
2      9.3
3      9.2
4      9.1
5      9.5
6     10.5
7     10.8
8     10.0
9      8.8
Name: lst_noche, dtype: float64


## Celda E4 · Días de calor extremo en la ciudad (MERRA-2, 2001–2025)

Temperatura máxima del aire de cada día en toda la ciudad. Un "día de calor extremo" es uno más caluroso que el 90 % de los días del mismo mes en 2001–2010.

Pide **un solo cálculo por mes** (con todos sus días juntos) para no repetir el error de "demasiadas consultas". Guarda después de cada año y se salta los años ya hechos.

In [12]:
ARCH_T = '/content/tmax_ciudad.csv'
t2m = ee.ImageCollection('NASA/GSFC/MERRA/slv/2').select('T2M')

def tmax_mes(ini):
    dias = list(pd.date_range(ini, ini + pd.offsets.MonthEnd(0)))
    img = ee.Image.cat([t2m.filterDate(d.strftime('%Y-%m-%d'), (d + pd.Timedelta(days=1)).strftime('%Y-%m-%d')).max()
                        .rename(d.strftime('d%Y%m%d')) for d in dias])
    vals = img.reduceRegion(ee.Reducer.mean(), municipio, 50000).getInfo()
    return [{'fecha': d, 'tmax': None if vals.get(d.strftime('d%Y%m%d')) is None else vals[d.strftime('d%Y%m%d')] - 273.15}
            for d in dias]

hechos = set()
if os.path.exists(ARCH_T):
    hechos = set(pd.read_csv(ARCH_T, parse_dates=['fecha'])['fecha'].dt.year)
for y in range(2001, 2026):
    if y in hechos:
        continue
    filas = []
    for ini in pd.date_range(f'{y}-01-01', f'{y}-12-01', freq='MS'):
        filas += reintentar(lambda: tmax_mes(ini), ini.strftime('%Y-%m'))
    pd.DataFrame(filas).to_csv(ARCH_T, mode='a', header=not os.path.exists(ARCH_T), index=False)
    print('Listo', y)

T = pd.read_csv(ARCH_T, parse_dates=['fecha']).drop_duplicates('fecha')
T['mes'] = T['fecha'].dt.month; T['anio'] = T['fecha'].dt.year
umbral = T[T['anio'] <= 2010].groupby('mes')['tmax'].quantile(0.9)          # umbral por mes
T['extremo'] = T['tmax'] > T['mes'].map(umbral)
D_calor = T.groupby('anio')['extremo'].sum()
print('Umbral por mes (°C):', umbral.round(1).to_dict())
print(D_calor)

Umbral por mes (°C): {1: 18.5, 2: 18.0, 3: 18.0, 4: 18.0, 5: 17.7, 6: 17.5, 7: 17.3, 8: 18.4, 9: 19.3, 10: 20.2, 11: 20.6, 12: 19.5}
anio
2001     19
2002     41
2003     51
2004     20
2005     60
2006     20
2007     31
2008     21
2009     39
2010     64
2011     12
2012     26
2013     32
2014     23
2015     43
2016     82
2017     40
2018     25
2019     36
2020     23
2021     25
2022     43
2023    105
2024     96
2025      5
Name: extremo, dtype: int64


In [13]:
print('Años con dato de Landsat por distrito:')
print(L.groupby('id')['lst'].count())
print('Ids del JSON:', sorted(clave(d['id']) for d in tj['distritos']))
print('Ids de Landsat:', sorted(L['id'].unique()))

Años con dato de Landsat por distrito:
id
1     13
10    13
11    13
12    13
13    13
14    13
15    13
2     13
3     13
4     13
5     13
6     13
7     13
8     13
9     13
Name: lst, dtype: int64
Ids del JSON: ['1', '10', '11', '12', '13', '14', '15', '2', '3', '4', '5', '6', '7', '8', '9']
Ids de Landsat: ['1', '10', '11', '12', '13', '14', '15', '2', '3', '4', '5', '6', '7', '8', '9']


## Celda E5 · Agregar todo al JSON

Al terminar, descarga `tendencias_distritos.json` y reemplaza el de `public/data/`. Descarga también los tres CSV de respaldo a `data/tendencias_modis/`. Guarda el cuaderno como `notebooks/calor_urbano.ipynb`.

In [14]:
por_id = {clave(d['id']): d for d in tj['distritos']}
for did, d in por_id.items():
    # calor nocturno dentro de la serie mensual
    sub = N[N['id'] == did]
    n = pd.Series(sub['lst_noche'].values, index=sub['fecha'].dt.strftime('%Y-%m'))
    for p in d['serie']:
        v = n.get(p['fecha']); p['lst_noche'] = None if v is None or pd.isna(v) else round(float(v), 2)
    d['tendencias']['lst_noche'] = analizar_mensual(sub[['fecha', 'lst_noche']])
    # calor Landsat como serie anual
    l = L[L['id'] == did].sort_values('anio')
    anual = {a['anio']: a for a in d.get('anual', [])}
    for r in l.itertuples():
        anual.setdefault(int(r.anio), {'anio': int(r.anio)}).update({
            'lst_landsat': None if pd.isna(r.lst) else round(float(r.lst), 2),
            'lst_landsat_rel': None if pd.isna(r.lst_rel) else round(float(r.lst_rel), 2)})
    d['anual'] = sorted(anual.values(), key=lambda a: a['anio'])
    d['tendencias']['lst_landsat'] = analizar_anual(l['anio'].values, l['lst'].values)
    d['tendencias']['lst_landsat_rel'] = analizar_anual(l['anio'].values, l['lst_rel'].values)

ciudad = tj.setdefault('ciudad', {'anual': [], 'tendencias': {}})
ca = {a['anio']: a for a in ciudad.get('anual', [])}
for a, v in D_calor.items():
    ca.setdefault(int(a), {'anio': int(a)})['dias_calor_extremo'] = int(v)
ciudad['anual'] = sorted(ca.values(), key=lambda a: a['anio'])
ciudad.setdefault('tendencias', {})['dias_calor_extremo'] = analizar_anual(D_calor.index.values, D_calor.values)

tj['meta'].update({
    'fuente_lst_landsat': 'Landsat 8 y 9 Collection 2 Nivel 2 (NASA/USGS), banda térmica, 100 m',
    'periodo_lst_landsat': '2013 a 2025, época seca (junio a octubre)',
    'nota_lst_landsat': 'Mediana de las escenas sin nubes de junio a octubre de cada año. Landsat pasa siempre cerca de las 10:30, así que los años son comparables.',
    'fuente_lst_noche': 'MODIS Aqua MYD11A2 v061 (LST nocturna, 1 km)', 'periodo_lst_noche': '2003-01 a 2021-12',
    'nota_lst_noche': 'Calor que el suelo conserva de noche. Se usa hasta 2021 porque la órbita de Aqua cambia desde 2022.',
    'fuente_calor_extremo': 'NASA MERRA-2 (temperatura máxima diaria del aire)',
    'nota_ciudad': 'Dato de toda la ciudad (resolución de unos 50 km o 10 km): no cambia entre distritos.',
    'definicion_calor_extremo': 'Día con temperatura máxima mayor que el 90 % de los días del mismo mes en 2001–2010.'})
tj['meta'].setdefault('unidades', {}).update({'lst_landsat': '°C/año', 'lst_landsat_rel': '°C/año', 'lst_noche': '°C/año', 'dias_calor_extremo': 'días/año'})

json.dump(tj, open('tendencias_distritos.json', 'w', encoding='utf-8'), ensure_ascii=False)
print(pd.DataFrame([{'distrito': d['nombre'],
                     'landsat_sen': d['tendencias']['lst_landsat']['sen_pendiente_anual'], 'landsat_p': d['tendencias']['lst_landsat']['mk_p'],
                     'rel_sen': d['tendencias']['lst_landsat_rel']['sen_pendiente_anual'], 'rel_p': d['tendencias']['lst_landsat_rel']['mk_p'],
                     'noche_sen': d['tendencias']['lst_noche']['sen_pendiente_anual'], 'noche_p': d['tendencias']['lst_noche']['mk_p']}
                    for d in tj['distritos']]))
print('Días de calor extremo:', ciudad['tendencias']['dias_calor_extremo'])

       distrito  landsat_sen  landsat_p  rel_sen   rel_p  noche_sen  noche_p
0    Distrito 1      -0.0638     0.6693   0.0068  0.8548     0.0473   0.0001
1   Distrito 10       0.0145     0.9514   0.0557  0.2997     0.0518   0.0002
2   Distrito 11      -0.0102     0.8548   0.0654  0.0327     0.0396   0.0031
3   Distrito 12      -0.0605     0.7603  -0.0249  0.8548     0.0751   0.0000
4   Distrito 13      -0.0443     0.7603   0.0292  0.9514     0.0355   0.0002
5   Distrito 14      -0.0954     0.2997  -0.0045  0.9514     0.0594   0.0000
6   Distrito 15      -0.0581     0.8548   0.0068  0.9514     0.0110   0.3077
7    Distrito 2      -0.1127     0.3601  -0.0553  0.4277     0.0671   0.0000
8    Distrito 3      -0.1334     0.3601  -0.1124  0.0769     0.0929   0.0000
9    Distrito 4      -0.1216     0.2997  -0.0725  0.2464     0.0767   0.0000
10   Distrito 5      -0.0110     0.9514   0.0618  0.3601     0.0479   0.0001
11   Distrito 6       0.0438     0.7603   0.1102  0.0041     0.0380   0.0051